# Blackjack Card Classifier — MobileNetV2 Transfer Learning

**Run this notebook on Google Colab with a GPU runtime** (Runtime → Change runtime type → T4 GPU).

This notebook trains a 52-class CNN to recognize playing cards from overhead corner-crop images.

### Steps:
1. Upload your `training_data/` folder to Colab (or mount Google Drive)
2. Run all cells
3. Download `model.tflite` and copy it to `rpi/model.tflite` on your Raspberry Pi

### Training data structure expected:
```
training_data/
  2_clubs/      (30 images)
  2_diamonds/   (30 images)
  ...           (52 folders total)
  A_spades/     (30 images)
```

Note: AI tools (Claude) were used to assist with code development.

In [ ]:
# Install dependencies
!pip install tensorflow -q

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import os

IMG_SIZE   = 64       # Match classify.py input size
BATCH_SIZE = 32
EPOCHS     = 20
NUM_CLASSES = 52

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ── Data loading with augmentation ───────────────────────────────────────────
# Augmentation: randomly flip, rotate ±10°, adjust brightness.
# This artificially expands the dataset and makes the model robust to
# minor lighting/angle variations — important for real-world reliability.

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    validation_split=0.15      # 15% of images held out for validation
)

train_gen = train_datagen.flow_from_directory(
    'training_data',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_gen = train_datagen.flow_from_directory(
    'training_data',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

print(f'Training samples:   {train_gen.samples}')
print(f'Validation samples: {val_gen.samples}')
print(f'Classes found:      {len(train_gen.class_indices)}')

# Save class index mapping (must match CARD_LABELS in classify.py)
import json
with open('class_indices.json', 'w') as f:
    json.dump(train_gen.class_indices, f, indent=2)
print('Class indices saved to class_indices.json')

In [ ]:
# ── Model architecture: MobileNetV2 + custom classification head ──────────────
#
# Transfer learning: MobileNetV2 was pre-trained on ImageNet (millions of images)
# so it already detects edges, textures, and shapes. We freeze those weights
# and add a new classification head trained on our 52-class card dataset.
# This is fast to train and accurate even with ~1,500 images.

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,       # Remove ImageNet classification head
    weights='imagenet'       # Load pre-trained weights
)
base_model.trainable = False  # Freeze base — only train the new head

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),                  # Prevents overfitting on small dataset
    layers.Dense(NUM_CLASSES, activation='softmax')  # 52-class output
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

val_acc = max(history.history['val_accuracy'])
print(f'\nBest validation accuracy: {val_acc:.1%}')

In [ ]:
# ── Export as TFLite ──────────────────────────────────────────────────────────
# TFLite is a compressed, optimized format for inference on embedded devices
# like the Raspberry Pi — much faster than running full TensorFlow.

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Quantization for speed
tflite_model = converter.convert()

with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

size_kb = os.path.getsize('model.tflite') / 1024
print(f'model.tflite saved ({size_kb:.0f} KB)')
print('Download model.tflite and copy it to rpi/model.tflite on your RPi.')